In [7]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# WorldCereal dataset
dataset = ee.ImageCollection('ESA/WorldCereal/2021/MODELS/v100')

def mask_other(img):
    return img.updateMask(img.neq(0))

dataset = dataset.map(mask_other)

# Barbados boundary
countries = ee.FeatureCollection('FAO/GAUL/2015/level0')

barbados = countries.filter(
    ee.Filter.eq('ADM0_NAME', 'Barbados')
)

# Temporary crops mosaic
temporarycrops = (
    dataset
    .filter(ee.Filter.eq('product', 'temporarycrops'))
    .mosaic()
    .clip(barbados)
)

# Maize product in Barbados (if available)
maize = (
    dataset
    .filter(ee.Filter.eq('product', 'maize'))
    .mosaic()
    .clip(barbados)
)

# Irrigation product in Barbados (if available)
irrigation = (
    dataset
    .filter(ee.Filter.eq('product', 'irrigation'))
    .mosaic()
    .clip(barbados)
)

# Display
Map = geemap.Map()

Map.centerObject(barbados, 11)

Map.addLayer(
    temporarycrops,
    {
        'bands': ['classification'],
        'min': 0,
        'max': 100,
        'palette': ['ff0000']
    },
    'Temporary Crops'
)

Map.addLayer(
    maize,
    {
        'bands': ['classification'],
        'min': 0,
        'max': 100,
        'palette': ['ebc334']
    },
    'Maize'
)

Map.addLayer(
    irrigation,
    {
        'bands': ['classification'],
        'min': 0,
        'max': 100,
        'palette': ['2d79eb']
    },
    'Irrigation'
)

# Map.addLayer(barbados, {'color': 'white'}, 'Barbados Boundary')

Map

Map(center=[13.172173693351903, -59.556391055409485], controls=(WidgetControl(options=['position', 'transparen…